In [1]:
import numpy as np
import tensorflow as tf
import tensorflow_hub as hub
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
import warnings
warnings.filterwarnings("ignore")

In [2]:
USE_URL       = "https://tfhub.dev/google/universal-sentence-encoder/4"   # DAN
# USE_URL     = "https://tfhub.dev/google/universal-sentence-encoder-large/5"  # Transformer

BATCH_SIZE    = 32
EPOCHS        = 10
LSTM_UNITS    = 100      # dl — matches paper
ATTN_UNITS    = 100      # da — matches paper
SHARED_UNITS  = 100      # ds — matches paper
LEARNING_RATE = 1e-4     # Adam lr — matches paper
LAMBDA        = 0.5      # loss weight for shared (eq.17)
TEST_SIZE     = 0.2
RANDOM_SEED   = 42

In [6]:
import pandas as pd
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# Download the VADER lexicon (only needs to be done once)
nltk.download('vader_lexicon')
analyzer = SentimentIntensityAnalyzer()

print("Loading dataset...")
df = pd.read_csv(r"C:\Users\chouh\OneDrive\Desktop\VirginiaTech\Spring 2026\SocialMediaAnalytics\FinalProject\archive\twitter_sentiment_data.csv")
df = df.dropna(subset=["message"])
df["message"] = df["message"].astype(str).str.strip()

# Sentiment Classification (Independent of Stance)

def get_vader_sentiment(text):
    # compound score ranges from -1 (Extremely Negative) to 1 (Extremely Positive)
    score = analyzer.polarity_scores(text)['compound']
    if score <= -0.05:
        return 0  # Negative
    elif score >= 0.05:
        return 2  # Positive
    else:
        return 1  # Neutral

# Apply VADER to the actual text content
df["sentiment_label"] = df["message"].apply(get_vader_sentiment)

# Stance: mapping based on the original Kaggle 'sentiment' column
# We keep this as is because 'sentiment' in this CSV is actually the 'Stance'
stance_df = df[df["sentiment"] != 0].copy()
stance_df["stance_label"] = stance_df["sentiment"].map({-1: 0, 1: 1, 2: 1})

# ── Summary Prints ──────────────────────────────────────────────────────
print(f"\nDataset : {len(df):,} total  |  {len(stance_df):,} for stance (neutral excluded)")
print("\n── Independent Sentiment (VADER) ──")
for k, n in {0: "Negative", 1: "Neutral", 2: "Positive"}.items():
    print(f"  {n:10s}: {(df['sentiment_label']==k).sum():,}")

print("\n── Stance (Kaggle Labels) ────────")
for k, n in {0: "Denier", 1: "Believer"}.items():
    print(f"  {n:10s}: {(stance_df['stance_label']==k).sum():,}")
print()

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\chouh\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


Loading dataset...

Dataset : 43,943 total  |  36,228 for stance (neutral excluded)

── Independent Sentiment (VADER) ──
  Negative  : 16,581
  Neutral   : 11,377
  Positive  : 15,985

── Stance (Kaggle Labels) ────────
  Denier    : 3,990
  Believer  : 32,238



In [9]:
#Load USE
use_model = hub.load(USE_URL)
print("USE loaded. \n")

def encode_texts(texts, batch_size=256):
    embeddings = []
    for i in range(0, len(texts), batch_size):
        emb = use_model(texts[i: i + batch_size]).numpy()
        embeddings.append(emb)
    return np.vstack(embeddings)

USE loaded. 



In [10]:
# USE 512-dim sentence vector. Reshape to (N, 1, 512) for BiLSTM.
X_stance = encode_texts(stance_df["message"].tolist())   # (N, 512)
X_stance_seq = np.expand_dims(X_stance, axis=1)          # (N, 1, 512)
print(f"Embedding shape: {X_stance.shape} :  BiLSTM input: {X_stance_seq.shape}\n")

y_stance    = stance_df["stance_label"].values
y_sentiment = stance_df["sentiment_label"].values

# Train / Test Split
(X_train, X_test,
 y_st_train, y_st_test,
 y_se_train, y_se_test) = train_test_split(
    X_stance_seq, y_stance, y_sentiment,
    test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y_stance)

Embedding shape: (36228, 512) :  BiLSTM input: (36228, 1, 512)



In [11]:
class SelfAttention(tf.keras.layers.Layer):
    """
    Scaled dot-product self-attention (eq. 1 in paper).
    SAj = softmax(Qj · Kj^T / sqrt(da)) · Vj
    """
    def __init__(self, units, name=None, **kwargs):
        super().__init__(name=name, **kwargs)
        self.units = units
        self.Wq = tf.keras.layers.Dense(units)
        self.Wk = tf.keras.layers.Dense(units)
        self.Wv = tf.keras.layers.Dense(units)

    def call(self, x):
        Q, K, V = self.Wq(x), self.Wk(x), self.Wv(x)
        scale   = tf.math.sqrt(tf.cast(self.units, tf.float32))
        weights = tf.nn.softmax(tf.matmul(Q, K, transpose_b=True) / scale, axis=-1)
        return tf.matmul(weights, V)


class InterAttention(tf.keras.layers.Layer):
    """
    Inter-attention (eq. 2,3 in paper):
    IAxy = softmax(Qx · Ky^T / sqrt(da)) · Vy
    Uses Q from source x and K,V from source y.
    """
    def __init__(self, units, name=None, **kwargs):
        super().__init__(name=name, **kwargs)
        self.units = units
        self.Wq = tf.keras.layers.Dense(units)
        self.Wk = tf.keras.layers.Dense(units)
        self.Wv = tf.keras.layers.Dense(units)

    def call(self, x, y):
        Q       = self.Wq(x)
        K, V    = self.Wk(y), self.Wv(y)
        scale   = tf.math.sqrt(tf.cast(self.units, tf.float32))
        weights = tf.nn.softmax(tf.matmul(Q, K, transpose_b=True) / scale, axis=-1)
        return tf.matmul(weights, V)


class GateSharingCell(tf.keras.layers.Layer):
    """
    Gate Sharing Cell (eq. 8,9 in paper):
      gd   = σ(Wd · Ashared + bd)
      Gd   = gd ⊙ Ashared
    Filters shared features relevant to a specific task.
    """
    def __init__(self, units, name=None, **kwargs):
        super().__init__(name=name, **kwargs)
        self.gate_dense = tf.keras.layers.Dense(units, activation="sigmoid")

    def call(self, A_task, A_shared):
        # Weights from A_task passed to gate, applied to A_shared
        gate = self.gate_dense(A_task)
        return gate * A_shared


class SPIALayer(tf.keras.layers.Layer):
    """
    Shared-Private Inter-Attention (eq. 11,12 in paper):
      SPIAd = softmax(Qd · Kshared^T) · Vshared
    Captures shared features relevant to a specific task.
    """
    def __init__(self, units, name=None, **kwargs):
        super().__init__(name=name, **kwargs)
        self.units  = units
        self.Wq     = tf.keras.layers.Dense(units)
        self.Wk     = tf.keras.layers.Dense(units)
        self.Wv     = tf.keras.layers.Dense(units)

    def call(self, A_task, A_shared):
        Q       = self.Wq(A_task)
        K, V    = self.Wk(A_shared), self.Wv(A_shared)
        scale   = tf.math.sqrt(tf.cast(self.units, tf.float32))
        weights = tf.nn.softmax(tf.matmul(Q, K, transpose_b=True) / scale, axis=-1)
        return tf.matmul(weights, V)


class FusionLayer(tf.keras.layers.Layer):
    """
    Fusion (eq. 13-16 in paper):
      C  = [G; SPIA; G-SPIA; G⊙SPIA]
      Fshared = tanh(Wf · C + bf)
    """
    def __init__(self, units, name=None, **kwargs):
        super().__init__(name=name, **kwargs)
        self.dense = tf.keras.layers.Dense(units, activation="tanh")

    def call(self, G, SPIA):
        C = tf.concat([G, SPIA, G - SPIA, G * SPIA], axis=-1)
        return self.dense(C)

# ── 7. Build SP-AMT Model ─────────────────────────────────────────────────────

def build_sp_amt():
    """
    Shared-Private Attention Multi-Task Model (SP-AMT).

    Since USE produces a single sentence vector (no token sequence),
    we use seq_len=1. Both private and shared BiLSTMs operate over
    this single time-step, and attention is applied over the hidden
    states.
    """
    inp = tf.keras.Input(shape=(1, 512), name="use_embedding")

    # ── Private BiLSTMs (task-specific feature extraction) ────────────────────
    bilstm_private_stance = tf.keras.layers.Bidirectional(
        tf.keras.layers.LSTM(LSTM_UNITS, return_sequences=True),
        name="bilstm_private_stance")(inp)                  # (B, 1, 2*dl)

    bilstm_private_sent = tf.keras.layers.Bidirectional(
        tf.keras.layers.LSTM(LSTM_UNITS, return_sequences=True),
        name="bilstm_private_sentiment")(inp)               # (B, 1, 2*dl)

    # ── Shared BiLSTM (task-invariant features) ───────────────────────────────
    bilstm_shared = tf.keras.layers.Bidirectional(
        tf.keras.layers.LSTM(LSTM_UNITS, return_sequences=True),
        name="bilstm_shared")(inp)                          # (B, 1, 2*dl)

    # Feature-Specific Attention (eq. 1-6)
    # For Stance (private):  SA + IA(private_stance ↔ shared)
    sa_stance  = SelfAttention(ATTN_UNITS, name="SA_stance")(bilstm_private_stance)
    ia_stance  = InterAttention(ATTN_UNITS, name="IA_stance_shared")(
                    bilstm_private_stance, bilstm_shared)
    Ad = tf.concat([sa_stance, ia_stance], axis=-1)         # eq. 5

    # For Sentiment (private): SA + IA(private_sent ↔ shared)
    sa_sent    = SelfAttention(ATTN_UNITS, name="SA_sentiment")(bilstm_private_sent)
    ia_sent    = InterAttention(ATTN_UNITS, name="IA_sentiment_shared")(
                    bilstm_private_sent, bilstm_shared)
    As = tf.concat([sa_sent, ia_sent], axis=-1)             # eq. 6

    #Shared Attention: Ashared = average(Ad, As)  (eq. 7)
    A_shared = tf.keras.layers.Average(name="A_shared")([Ad, As])

    #Gate Sharing Cell (eq. 8,9,10)
    Gd = GateSharingCell(SHARED_UNITS * 2, name="gate_stance")(Ad, A_shared)
    Gs = GateSharingCell(SHARED_UNITS * 2, name="gate_sentiment")(As, A_shared)

    #SPIA (eq. 11,12)
    SPIAd = SPIALayer(SHARED_UNITS * 2, name="SPIA_stance")(Ad, A_shared)
    SPIAs = SPIALayer(SHARED_UNITS * 2, name="SPIA_sentiment")(As, A_shared)

    # Fusion (eq. 13-16)
    Fd_shared = FusionLayer(SHARED_UNITS * 2, name="fusion_stance")(Gd, SPIAd)
    Fs_shared = FusionLayer(SHARED_UNITS * 2, name="fusion_sentiment")(Gs, SPIAs)

    #Classification Layer (eq. 17 — final concat of private + shared)
    # Stance: concat(Ad, Fd_shared)
    stance_feat = tf.keras.layers.Concatenate(name="stance_features")([Ad, Fd_shared])
    stance_flat = tf.keras.layers.Flatten(name="stance_flat")(stance_feat)
    stance_h    = tf.keras.layers.Dense(64, activation="relu")(stance_flat)
    stance_h    = tf.keras.layers.Dropout(0.3)(stance_h)
    out_stance  = tf.keras.layers.Dense(1, activation="sigmoid",
                                        name="stance_output")(stance_h)

    # Sentiment: concat(As, Fs_shared)
    sent_feat   = tf.keras.layers.Concatenate(name="sentiment_features")([As, Fs_shared])
    sent_flat   = tf.keras.layers.Flatten(name="sentiment_flat")(sent_feat)
    sent_h      = tf.keras.layers.Dense(64, activation="relu")(sent_flat)
    sent_h      = tf.keras.layers.Dropout(0.3)(sent_h)
    out_sent    = tf.keras.layers.Dense(3, activation="softmax",
                                        name="sentiment_output")(sent_h)

    return tf.keras.Model(inputs=inp, outputs=[out_stance, out_sent], name="SP_AMT_USE")


In [12]:
#Compile & Train
print("Building SP-AMT with USE embeddings")

model = build_sp_amt()
model.summary()

model.compile(
    optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
    loss={
        "stance_output":    "binary_crossentropy",        # primary task
        "sentiment_output": "sparse_categorical_crossentropy"  # auxiliary
    },
    loss_weights={
        "stance_output":    1.0,       # Ltask
        "sentiment_output": LAMBDA     # λ·Lshared  (eq. 17)
    },
    metrics={
        "stance_output":    ["accuracy"],
        "sentiment_output": ["accuracy"]
    }
)

history = model.fit(
    X_train,
    {"stance_output": y_st_train, "sentiment_output": y_se_train},
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    verbose=1
)

Building SP-AMT with USE embeddings
Model: "SP_AMT_USE"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 use_embedding (InputLayer)  [(None, 1, 512)]             0         []                            
                                                                                                  
 bilstm_private_stance (Bid  (None, 1, 200)               490400    ['use_embedding[0][0]']       
 irectional)                                                                                      
                                                                                                  
 bilstm_shared (Bidirection  (None, 1, 200)               490400    ['use_embedding[0][0]']       
 al)                                                                                              
                                                     

In [14]:
# Evaluate
print("Evaluation on Test Set")

pred_stance_raw, pred_sent_raw = model.predict(X_test)
pred_stance = (pred_stance_raw > 0.5).astype(int).flatten()
pred_sent   = np.argmax(pred_sent_raw, axis=1)

print(" Stance Detection  (Primary Task) ")
print(" Classes: Denier (0) vs Believer (1)")
print(classification_report(y_st_test, pred_stance,
                             target_names=["Denier", "Believer"]))
f1_stance = f1_score(y_st_test, pred_stance, average="macro")

print("Sentiment Analysis  (Auxiliary Task) ")
print("   Classes: Negative (0) | Neutral (1) | Positive (2)")
print(classification_report(y_se_test, pred_sent,
                             target_names=["Negative", "Neutral", "Positive"]))
f1_sent = f1_score(y_se_test, pred_sent, average="macro")

Evaluation on Test Set
227/227 [==============================] - 2s 7ms/step
 Stance Detection  (Primary Task) 
 Classes: Denier (0) vs Believer (1)
              precision    recall  f1-score   support

      Denier       0.66      0.38      0.48       798
    Believer       0.93      0.98      0.95      6448

    accuracy                           0.91      7246
   macro avg       0.79      0.68      0.72      7246
weighted avg       0.90      0.91      0.90      7246

Sentiment Analysis  (Auxiliary Task) 
   Classes: Negative (0) | Neutral (1) | Positive (2)
              precision    recall  f1-score   support

    Negative       0.66      0.78      0.71      2853
     Neutral       0.67      0.51      0.58      1899
    Positive       0.64      0.63      0.63      2494

    accuracy                           0.66      7246
   macro avg       0.66      0.64      0.64      7246
weighted avg       0.66      0.66      0.65      7246

